In [ ]:
# ============================================================
# Textile Productivity Prediction
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

os.makedirs("images", exist_ok=True)
os.makedirs("models", exist_ok=True)

In [ ]:
# ============================================================
# 1. Load Dataset
# ============================================================

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00597/garments_worker_productivity.csv"
df = pd.read_csv(url)

df.head()

In [ ]:
# ============================================================
# 2. Basic Cleaning
# ============================================================

df.columns = df.columns.str.strip()
df["department"] = df["department"].astype(str).str.strip().str.lower()
df["date"] = pd.to_datetime(df["date"], errors="coerce")

df = df.dropna(subset=["actual_productivity", "date"])

df["department"].value_counts()

In [ ]:
# ============================================================
# 3. Feature Engineering
# ============================================================

def create_features(data):
    data = data.copy()

    data["ordinal_date"] = data["date"].map(lambda x: x.toordinal())

    data["theoretical_productivity"] = data["smv"] * data["no_of_workers"]
    data["incentive_ratio"] = data["incentive"] / (data["over_time"] + 1)
    data["workload_per_worker"] = data["smv"] / (data["no_of_workers"] + 1)
    data["time_per_worker"] = data["over_time"] / (data["no_of_workers"] + 1)
    data["style_change_rate"] = data["no_of_style_change"] / (data["no_of_workers"] + 1)

    data = data.drop(columns=["date"], errors="ignore")

    return data

In [ ]:
# ============================================================
# 4. Evaluation Function
# ============================================================

def evaluate_model(y_true, y_pred):
    return {
        "R2": round(r2_score(y_true, y_pred), 4),
        "MAE": round(mean_absolute_error(y_true, y_pred), 4),
        "RMSE": round(np.sqrt(mean_squared_error(y_true, y_pred)), 4)
    }

In [ ]:
# ============================================================
# 5. Training Function by Department
# ============================================================

def train_department_model(df, department_name):
    data_dept = df[df["department"] == department_name].copy()
    data_dept = create_features(data_dept)

    target = "actual_productivity"

    X = data_dept.drop(columns=[target])
    y = data_dept[target]

    categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
    numerical_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, numerical_cols),
        ("cat", categorical_pipeline, categorical_cols)
    ])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    models = {
        "Linear Regression": LinearRegression(),
        "Random Forest": RandomForestRegressor(
            n_estimators=200,
            random_state=42,
            max_depth=8
        ),
        "Gradient Boosting": GradientBoostingRegressor(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=3,
            random_state=42
        )
    }

    results = []
    trained_models = {}

    for model_name, model in models.items():
        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)

        metrics = evaluate_model(y_test, y_pred)

        results.append({
            "Department": department_name,
            "Model": model_name,
            **metrics
        })

        trained_models[model_name] = {
            "pipeline": pipeline,
            "y_test": y_test,
            "y_pred": y_pred
        }

    return pd.DataFrame(results), trained_models

In [ ]:
# ============================================================
# 6. Train Models
# ============================================================

sewing_results, sewing_models = train_department_model(df, "sweing")
finishing_results, finishing_models = train_department_model(df, "finishing")

results = pd.concat([sewing_results, finishing_results], ignore_index=True)

results

In [ ]:
# Save results for README
results.to_csv("model_results.csv", index=False)

In [ ]:
# ============================================================
# 7. Select Best Model per Department
# ============================================================

best_results = results.sort_values(["Department", "R2"], ascending=[True, False])
best_results

In [ ]:
best_sewing_model_name = sewing_results.sort_values("R2", ascending=False).iloc[0]["Model"]
best_finishing_model_name = finishing_results.sort_values("R2", ascending=False).iloc[0]["Model"]

best_sewing_model = sewing_models[best_sewing_model_name]
best_finishing_model = finishing_models[best_finishing_model_name]

print("Best Sewing Model:", best_sewing_model_name)
print("Best Finishing Model:", best_finishing_model_name)

In [ ]:
# ============================================================
# 8. Predicted vs Actual Plot
# ============================================================

def plot_predicted_vs_actual(y_test, y_pred, title, filename):
    plt.figure(figsize=(7, 6))
    plt.scatter(y_test, y_pred, alpha=0.7)
    plt.plot(
        [y_test.min(), y_test.max()],
        [y_test.min(), y_test.max()],
        linestyle="--"
    )
    plt.xlabel("Actual Productivity")
    plt.ylabel("Predicted Productivity")
    plt.title(title)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"images/{filename}", dpi=300)
    plt.show()

In [ ]:
plot_predicted_vs_actual(
    best_sewing_model["y_test"],
    best_sewing_model["y_pred"],
    f"Sewing - Predicted vs Actual ({best_sewing_model_name})",
    "sewing_predicted_vs_actual.png"
)

plot_predicted_vs_actual(
    best_finishing_model["y_test"],
    best_finishing_model["y_pred"],
    f"Finishing - Predicted vs Actual ({best_finishing_model_name})",
    "finishing_predicted_vs_actual.png"
)

In [ ]:
# ============================================================
# 9. Residual Plot
# ============================================================

def plot_residuals(y_test, y_pred, title, filename):
    residuals = y_test - y_pred

    plt.figure(figsize=(7, 6))
    plt.scatter(y_pred, residuals, alpha=0.7)
    plt.axhline(0, linestyle="--")
    plt.xlabel("Predicted Productivity")
    plt.ylabel("Residuals")
    plt.title(title)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"images/{filename}", dpi=300)
    plt.show()

In [ ]:
plot_residuals(
    best_sewing_model["y_test"],
    best_sewing_model["y_pred"],
    f"Sewing - Residual Analysis ({best_sewing_model_name})",
    "sewing_residuals.png"
)

plot_residuals(
    best_finishing_model["y_test"],
    best_finishing_model["y_pred"],
    f"Finishing - Residual Analysis ({best_finishing_model_name})",
    "finishing_residuals.png"
)

In [ ]:
# ============================================================
# 10. Feature Importance
# ============================================================

def plot_feature_importance(pipeline, title, filename, top_n=15):
    model = pipeline.named_steps["model"]
    preprocessor = pipeline.named_steps["preprocessor"]

    if not hasattr(model, "feature_importances_"):
        print("This model does not provide feature importance.")
        return

    feature_names = preprocessor.get_feature_names_out()
    importances = model.feature_importances_

    importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    }).sort_values("importance", ascending=False).head(top_n)

    plt.figure(figsize=(9, 6))
    plt.barh(importance_df["feature"][::-1], importance_df["importance"][::-1])
    plt.xlabel("Feature Importance")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(f"images/{filename}", dpi=300)
    plt.show()

    return importance_df

In [ ]:
sewing_importance = plot_feature_importance(
    best_sewing_model["pipeline"],
    "Top Feature Importance - Sewing",
    "sewing_feature_importance.png"
)

finishing_importance = plot_feature_importance(
    best_finishing_model["pipeline"],
    "Top Feature Importance - Finishing",
    "finishing_feature_importance.png"
)

In [ ]:
# ============================================================
# 11. Save Models
# ============================================================

joblib.dump(best_sewing_model["pipeline"], "models/best_sewing_model.pkl")
joblib.dump(best_finishing_model["pipeline"], "models/best_finishing_model.pkl")

In [ ]:
# ============================================================
# 12. Simple Inference Function
# ============================================================

def predict_productivity(input_df, department):
    """
    Predict productivity for new records.

    Parameters:
        input_df: pandas DataFrame with raw input features.
        department: "sweing" or "finishing"

    Returns:
        Predicted productivity values.
    """

    if department == "sweing":
        model = joblib.load("models/best_sewing_model.pkl")
    elif department == "finishing":
        model = joblib.load("models/best_finishing_model.pkl")
    else:
        raise ValueError("department must be 'sweing' or 'finishing'")

    input_df = create_features(input_df)

    return model.predict(input_df)

In [ ]:
results_sorted = results.sort_values(["Department", "R2"], ascending=[True, False])
print(results_sorted)

In [ ]:
print("\n=== Interpretation ===")
print("Sewing department shows strong predictive performance.")
print("Finishing department shows weak performance due to low signal in features.")

In [ ]:
plt.figure(figsize=(6,4))
df[df["department"] == "finishing"]["actual_productivity"].hist(bins=20)
plt.title("Finishing Productivity Distribution")
plt.xlabel("Productivity")
plt.ylabel("Frequency")
plt.grid(True)
plt.savefig("images/finishing_distribution.png")
plt.show()

In [ ]:
corr = df.select_dtypes(include=[np.number]).corr()["actual_productivity"].sort_values(ascending=False)
print(corr.head(10))

In [ ]:
!pip install shap -q

import shap
import matplotlib.pyplot as plt

# Get raw Sewing data
sewing_raw = df[df["department"] == "sweing"].copy()

# Apply the same feature engineering used during training
sewing_features = create_features(sewing_raw)

# Remove target column
X_sewing = sewing_features.drop("actual_productivity", axis=1)

# Access trained pipeline parts
preprocessor = best_sewing_model["pipeline"].named_steps["preprocessor"]
model = best_sewing_model["pipeline"].named_steps["model"]

# Transform features
X_sewing_processed = preprocessor.transform(X_sewing)

# Get transformed feature names
feature_names = preprocessor.get_feature_names_out()

# Use a sample to keep SHAP fast
X_sample = X_sewing_processed[:200]

# Create SHAP explainer
explainer = shap.Explainer(model, X_sample)

# Calculate SHAP values
shap_values = explainer(X_sample)

# Plot SHAP summary
shap.summary_plot(
    shap_values,
    X_sample,
    feature_names=feature_names,
    show=False
)

plt.tight_layout()
plt.savefig("images/sewing_shap_summary.png", dpi=300, bbox_inches="tight")
plt.show()